# Data Ingestion & Validation

The **objective** of this notebook is to understand the structure,
quality, completeness, and potential issues within the raw retail
transaction dataset before making any preprocessing decisions.

It examines the data types, missing values, duplicate records, cancelled transactions, negativequantities, invalid prices etc.

The findings from this stage will be used to make evidence-based
preprocessing decisions and create reliable datasets for subsequent
sales, product, customer, and RFM analysis.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

# When you display a DataFrame, don't hide any columns.
# Display up to 100 rows when showing a DataFrame."
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
# Load the raw dataset
PROJECT_DIR = Path.cwd().parent
RAW_FILE = PROJECT_DIR / "data" / "raw" / "Online Retail.xlsx"

df = pd.read_excel(RAW_FILE)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
df.shape

(541909, 8)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [12]:
# Missing values
missing = df.isnull().sum()
missing

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [13]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage.round(2)
})

missing_summary

,Missing_Count,Missing_Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


In [15]:
# Duplicate records
duplicate_count  = df.duplicated().sum()
print(duplicate_count)

print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(df) * 100:.2f}%"
)

5268
Duplicate percentage: 0.97%


In [17]:
# Unique customers
print("Unique customers:", df["CustomerID"].nunique())
print("Missing Customer IDs:", df["CustomerID"].isna().sum())

Unique customers: 4372
Missing Customer IDs: 135080


In [18]:
# Unique products
print("Unique products:", df["StockCode"].nunique())
print("Unique product descriptions:", df["Description"].nunique())

Unique products: 4070
Unique product descriptions: 4223


In [22]:
# Date range
print("Minimum transaction date:",df["InvoiceDate"].min())
print("\nMaximum transaction date:",df["InvoiceDate"].max())

Minimum transaction date: 2010-12-01 08:26:00

Maximum transaction date: 2011-12-09 12:50:00


In [23]:
df["Quantity"].describe()

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

- `min` is negative, that's not automatically an error.

- It could represent: returned goods / cancelled transactions

In [24]:
df["UnitPrice"].describe()

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

In [26]:
# Negative quantities
negative_quantity = df[df["Quantity"] < 0]

print(f"Rows with negative quantity: {len(negative_quantity):,}")
negative_quantity.head()

Rows with negative quantity: 10,624


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [28]:
# Cancelled Invoices
cancelled = df[df["InvoiceNo"].astype(str).str.startswith("C")]
print(f"Cancelled invoice rows: {len(cancelled):,}")
cancelled.head()

Cancelled invoice rows: 9,288


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


## Investigating Data Quality Issues

Before preprocessing the dataset, we investigate the meaning and
business impact of missing values, duplicates, negative quantities,
and cancelled transactions.

The objective is to make evidence-based preprocessing decisions
rather than removing records blindly.

In [30]:
# Check zero and negative prices
zero_price = df[df["UnitPrice"] <= 0]

print(f"Transactions with UnitPrice <= 0: {len(zero_price):,}")
zero_price.head()

Transactions with UnitPrice <= 0: 2,517


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


A transaction with:
- Quantity = 5
- UnitPrice = 0
  
would give:
- Sales = 0

In [31]:
# Determine whether negative quantities and cancellation invoices are essentially describing the same transactions
negative_qty = df["Quantity"] < 0
cancelled_invoice = df["InvoiceNo"].astype(str).str.startswith("C")

print("Negative quantity rows:", negative_qty.sum())
print("Cancelled invoice rows:", cancelled_invoice.sum())
print(
    "Negative quantity + cancelled invoice:",
    (negative_qty & cancelled_invoice).sum()
)

Negative quantity rows: 10624
Cancelled invoice rows: 9288
Negative quantity + cancelled invoice: 9288


- All 9,288 cancelled-invoice rows are negative-quantity transactions.
- 87.4% of negative-quantity rows are associated with cancelled invoices.

```
1,336 negative-quantity rows aren't identified as cancelled invoices.
So, negative quantity can represent a return / reversal.
```

In [33]:
# Inspect non-cancelled negative quantities
negative_not_cancelled = df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))]

print(
    f"Negative quantity without cancellation code: "
    f"{len(negative_not_cancelled):,}"
)
negative_not_cancelled.head()

Negative quantity without cancellation code: 1,336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom


In [35]:
# Impact of missing CustomerID
customer_missing = df["CustomerID"].isna()

print("Total transactions:", f"{len(df):,}")
print("Transactions with CustomerID:", f"{(~customer_missing).sum():,}")
print("Transactions without CustomerID:", f"{customer_missing.sum():,}")

print(
    f"Percentage without CustomerID: "
    f"{customer_missing.mean() * 100:.2f}%"
)

Total transactions: 541,909
Transactions with CustomerID: 406,829
Transactions without CustomerID: 135,080
Percentage without CustomerID: 24.93%


In [37]:
# Revenue impact of missing CustomerID
df["Sales"] = df["Quantity"] * df["UnitPrice"]

missing_customer_sales = df.loc[df["CustomerID"].isna(), "Sales"].sum()

known_customer_sales = df.loc[df["CustomerID"].notna(), "Sales"].sum()

total_sales = df["Sales"].sum()

print(f"Total sales: ₹{total_sales:,.2f}")
print(f"Sales with CustomerID: ₹{known_customer_sales:,.2f}")
print(f"Sales without CustomerID: ₹{missing_customer_sales:,.2f}")

print(
    f"\nPercentage of sales without CustomerID: "
    f"{missing_customer_sales / total_sales * 100:.2f}%"
)

Total sales: ₹9,747,747.93
Sales with CustomerID: ₹8,300,065.81
Sales without CustomerID: ₹1,447,682.12

Percentage of sales without CustomerID: 14.85%


In [38]:
# Duplicate investigation
# keep=False --> Show me every row that belongs to a group of duplicate rows, including the 1st occurrence.
duplicates = df[df.duplicated(keep=False)]
print(f"Duplicate records: {len(duplicates):,}")

Duplicate records: 10,147


In [39]:
# Invoice-level investigation
print("Unique invoices:", df["InvoiceNo"].nunique())
print("Unique customers:", df["CustomerID"].nunique())
print("Unique products:", df["StockCode"].nunique())
print("Unique countries:", df["Country"].nunique())

Unique invoices: 25900
Unique customers: 4372
Unique products: 4070
Unique countries: 38
